# v0.11.0 -- JSON Patch & atomic field/array ops

Low-level, concurrency-safe writes -- each compiles to **one atomic server-side `UPDATE`**, so
parallel callers don't clobber each other. **Identical on SurrealDB 2.6.x and 3.x by design.**

- **`model.patch(operations)`** -- apply a JSON Patch (RFC 6902) to one record (needs an id).
- **Atomic helpers** -- `atomic_append` (array, dups ok), `atomic_set_add` (array, no dup),
  `atomic_remove` (removes *all* occurrences), `atomic_increment` (numeric, default `+1`,
  supports nested dotted paths).
- **`QuerySet.patch(operations)`** -- patch every matching row; returns the affected count.

These primitives emit **no signals** (unlike `merge`/`save`); reach for `merge()`/`save()` when
you need lifecycle hooks. All accept `tx=`.

## 1. Connect

In [1]:
import os
from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
# WebSocket (.../rpc) lets transaction() use SurrealDB 3.x native interactive transactions.
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root", password="root",
    namespace="examples", database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 2. Model and reset

In [2]:
import contextlib

from surreal_orm_lite import BaseSurrealModel, SurrealConfigDict


class Article(BaseSurrealModel):
    model_config = SurrealConfigDict(primary_key="id")
    id: str | None = None
    title: str = ""
    age: int = 0
    tags: list[str] = []
    views: int = 0
    score: float = 0.0
    counters: dict[str, int] = {}


client = await SurrealDBConnectionManager.get_client()
with contextlib.suppress(Exception):
    await client.query("DELETE Article;", {})
print("table reset")

table reset


## 3. `patch()` -- replace / add / remove on one record

A non-transactional `patch()` applies the server's returned row back to the instance, so `self` stays in sync.

In [3]:
a = Article(id="a1", title="Hello", age=20, tags=["intro"])
await a.save()

await a.patch([
    {"op": "replace", "path": "/age", "value": 26},
    {"op": "add", "path": "/tags/-", "value": "premium"},
    {"op": "add", "path": "/title", "value": "Hello, world"},
])
print("instance synced:", a.age, a.tags, "|", a.title)
print("in DB:", await client.query("SELECT age, tags, title FROM Article:a1;", {}))

instance synced: 26 ['intro', 'premium'] | Hello, world
in DB: [{'age': 26, 'tags': ['intro', 'premium'], 'title': 'Hello, world'}]


## 4. Atomic array ops

`append` keeps duplicates; `set_add` is set-like (no duplicate); `remove` deletes **every** occurrence -- the same on both DB lines.

In [4]:
p = Article(id="arr", tags=["x"])
await p.save()

await p.atomic_append("tags", "x")     # duplicate allowed
await p.atomic_append("tags", "y")
print("after append:", p.tags)         # ['x', 'x', 'y']

await p.atomic_set_add("tags", "x")    # already present -> unchanged
await p.atomic_set_add("tags", "z")    # absent -> added
print("after set_add:", p.tags)        # ['x', 'x', 'y', 'z']

await p.atomic_remove("tags", "x")     # removes ALL 'x'
print("after remove:", p.tags)         # ['y', 'z']

after append: ['x', 'x', 'y']
after set_add: ['x', 'x', 'y', 'z']
after remove: ['y', 'z']


### Batch variants -- apply many in one round-trip

`atomic_append_many` / `atomic_set_add_many` / `atomic_remove_many` take a list and compile to `array::concat` / `array::add` / `array::complement` (an empty list is a safe no-op).

In [5]:
b = Article(id="batch", tags=["a", "b"])
await b.save()

await b.atomic_append_many("tags", ["c", "d", "a"])   # all appended, dup 'a' kept
print("append_many:", b.tags)                          # ['a', 'b', 'c', 'd', 'a']

await b.atomic_set_add_many("tags", ["a", "e"])        # 'a' present -> only 'e' added
print("set_add_many:", b.tags)

await b.atomic_remove_many("tags", ["a", "c"])         # removes ALL 'a' and 'c'
print("remove_many:", b.tags)                          # ['b', 'd', 'e']

append_many: ['a', 'b', 'c', 'd', 'a']
set_add_many: ['a', 'b', 'c', 'd', 'a', 'e']
remove_many: ['b', 'd', 'e']


## 5. Atomic numeric increment (incl. nested paths)

`atomic_increment` defaults to `+1`, takes any amount (negative to decrement), and accepts a dotted path into a nested object.

In [6]:
c = Article(id="cnt", views=10, counters={"likes": 0})
await c.save()

await c.atomic_increment("views")         # +1 (default)
await c.atomic_increment("views", 5)      # +5
await c.atomic_increment("views", -6)     # -6
print("views:", c.views)                  # 10

await c.atomic_increment("counters.likes", 3)  # nested dotted path
print("counters:", c.counters)                 # {'likes': 3}

views: 10
counters: {'likes': 3}


## 6. `QuerySet.patch()` -- patch a filtered set (or the whole table)

Returns how many rows were affected.

In [7]:
with contextlib.suppress(Exception):
    await client.query("DELETE Article;", {})
await Article(id="q1", title="A", age=1).save()
await Article(id="q2", title="A", age=1).save()
await Article(id="q3", title="B", age=1).save()

n = await Article.objects().filter(title="A").patch([{"op": "replace", "path": "/age", "value": 99}])
print("rows patched (title='A'):", n)

rows = await client.query("SELECT id, age FROM Article ORDER BY id;", {})
print({str(r["id"]): r["age"] for r in rows})

rows patched (title='A'): 2
{'Article:q1': 99, 'Article:q2': 99, 'Article:q3': 1}


## 7. Combine `patch()` and an atomic op in one transaction

Both land atomically; a rollback would undo both together.

In [8]:
t = Article(id="t1", age=1, score=0.0)
await t.save()

async with SurrealDBConnectionManager.transaction() as tx:
    await t.patch([{"op": "replace", "path": "/age", "value": 7}], tx=tx)
    await t.atomic_increment("score", 2.0, tx=tx)

print(await client.query("SELECT age, score FROM Article:t1;", {}))

[{'age': 7, 'score': 2.0}]


## 8. Fast, clear validation errors

`patch()` validates the operations before any I/O, so a malformed patch fails immediately with a helpful message.

In [9]:
# Validation runs inside the public API, synchronously, BEFORE any I/O —
# a malformed patch is rejected without touching the database.
for bad in ([], [{"op": "frobnicate", "path": "/x", "value": 1}], [{"op": "add", "path": "/x"}]):
    try:
        await a.patch(bad)
    except ValueError as exc:
        print("rejected:", exc)

# A bad JSON Pointer (must be empty or start with '/') is rejected the same way:
try:
    await a.patch([{"op": "replace", "path": "age", "value": 1}])
except ValueError as exc:
    print("rejected pointer:", exc)

rejected: patch operations must be a non-empty list of operation dicts
rejected: patch operation #0 has invalid op 'frobnicate'; expected one of ['add', 'change', 'copy', 'move', 'remove', 'replace', 'test']
rejected: patch operation #0 ('add') is missing required 'value'
rejected pointer: Invalid operation #0 path 'age': a JSON Pointer must be empty or start with '/'


## 9. Cleanup

In [10]:
await client.query("DELETE Article;", {})
await SurrealDBConnectionManager.close_connection()
print("done")

done
